# Step 05 – Fine-tune and evaluate BERT on UC1 (Google Colab)

Fine-tunes `bert-base-uncased` on the same UC1 train and validation splits as the TF-IDF models and GPT-4o.
It scores the full test set and the shared 400-email sample, then draws the attention figures.
It runs on Colab because fine-tuning needs a GPU; a T4 is enough.

Thesis: Chapter 3, Sections 3.4–3.5; Chapter 4, Section 4.1 (Tables 2–3) and Section 4.3 (Figures 5–6).

Reads: `READ_DIR/train.csv`, `val.csv`, `test.csv` — UC1 splits created by `02_prepare_data.py`.

Writes:  
`SAVE_DIR/bert_uc1_full.csv` — metrics on the full test set (Table 2).  
`SAVE_DIR/bert_uc1_400.csv` — metrics on the 400-email sample (Table 3).  
`SAVE_DIR/bert_predictions.csv` — prediction for each test email, used by `11_plot_confusion_matrices.py`.  
`SAVE_DIR/bert_uc1_results.zip` — the three CSV files, downloaded to your computer.  
`HEATMAP_DIR/attention_*.png` and `.html` — attention figures (Figures 5–6).  
`MODEL_DIR/` — the fine-tuned model, only if `SAVE_MODEL = True`.

Usage: run the cells from top to bottom on a GPU runtime.

Options:  
`SMOKE_TEST` (section 3) — `True` for a quick check on a small subset; `False` for the real run.

## How this notebook works

**Evaluation.** BERT is scored on the full test set of 13,030 emails (Table 2) and on the same
400-email sample as GPT-4o, drawn with the same seed and code as `04_run_uc1_llm.py` (Table 3).

**Folders.** Colab cannot import `config.py`, so the folders are set at the top of section 2.
By default it reads from `MyDrive/Thesis/processed/` and writes the CSV files to `/content`.

**Running it.**
1. Use a GPU runtime (Runtime → Change runtime type → **T4 GPU**). A full run takes about 30-45 minutes.
2. Upload `train.csv`, `val.csv` and `test.csv` to `READ_DIR`. `train.csv` is about 145 MB, so the Drive website or app is more reliable than the Colab browser.
3. Run once with `SMOKE_TEST = True`, then with `False` for the real results.
4. Put the downloaded CSV files (section 11) in `RESULTS_DIR` (by default `Thesis Codes/results/`); step 11 needs `bert_predictions.csv` there.


## 1. Install / import

In [ ]:
# Colab already has torch + CUDA; install the Hugging Face stack to match requirements.txt.
!pip -q install "transformers>=4.40" "datasets>=2.18" accelerate scikit-learn --upgrade


In [ ]:
from __future__ import annotations
import json as _json
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — go to Runtime > Change runtime type > T4 GPU, then re-run.")


## 2. Folders and data splits

The first cell sets the folders. The splits are the files from `02_prepare_data.py`. They are not
rebuilt here, so BERT is tested on the same emails as the other UC1 models.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folders used by this notebook (Colab cannot import config.py; change them here).
# "/content" is the Colab session disk and is deleted when the session ends.
READ_DIR = Path('/content/drive/MyDrive/Thesis/processed')            # train/val/test.csv (from 02)
SAVE_DIR = Path('/content')                                          # result CSV and ZIP files
CHECKPOINT_DIR = Path('/content/bert_uc1_ckpt')                      # temporary Trainer checkpoints
MODEL_DIR = Path('/content/drive/MyDrive/Thesis/models/bert_uc1')    # only if SAVE_MODEL = True
# Attention figures (section 12).
HEATMAP_DIR = Path('/content/drive/MyDrive/Thesis/results/attention_heatmaps')

REQUIRED = ['train.csv', 'val.csv', 'test.csv']
missing = [f for f in REQUIRED if not (READ_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} under {READ_DIR}. Upload Thesis Codes/data/processed/{{train,val,test}}.csv "
        "to your Drive (matching this path, or edit READ_DIR above) and re-run this cell."
    )
print("Found all required files in", READ_DIR)


In [ ]:
RANDOM_SEED = 42  # same as config.RANDOM_SEED; needed for the 400-email sample

def load_split(name: str) -> pd.DataFrame:
    """Load one UC1 split and fill blanks in the columns the model needs."""
    df = pd.read_csv(READ_DIR / f"{name}.csv")
    for col in ("text", "sender_tld"):
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)
    if "num_urls" in df.columns:
        df["num_urls"] = pd.to_numeric(df["num_urls"], errors="coerce").fillna(0)
    df["label"] = df["label"].astype(int)
    return df

train_df = load_split("train")
val_df   = load_split("val")
test_df  = load_split("test").reset_index(drop=True)  # keeps row numbers aligned with step 04

print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print("train label balance:\n", train_df['label'].value_counts())

# Split sizes from Section 3.5; a mismatch usually means an old or partial upload.
expected = {"train": 45600, "val": 6514, "test": 13030}
for name, df, exp in (("train", train_df, expected["train"]),
                       ("val", val_df, expected["val"]),
                       ("test", test_df, expected["test"])):
    if len(df) != exp:
        print(f"  NOTE: {name}.csv has {len(df):,} rows, the thesis reports {exp:,}. "
              f"Not necessarily wrong (pipeline may have been re-run), but worth double-checking.")


## 3. Config

`bert-base-uncased` is the comparison model of Section 3.4. 256 tokens cover most emails;
longer ones are truncated, as in the GPT-4o runner.


In [ ]:
CONFIG = {
    "model_name": "bert-base-uncased",
    "max_len": 256,
    "batch_size": 16,
    "eval_batch_size": 32,
    "epochs": 3,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "seed": RANDOM_SEED,
    "fp16": torch.cuda.is_available(),
}

# True = quick check on a small subset (1 epoch, 2-3 minutes); False = full run for the thesis.
SMOKE_TEST = True

print(CONFIG)


## 4. Tokenize

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=CONFIG["max_len"], padding="max_length")

def prep(df: pd.DataFrame) -> Dataset:
    ds = to_hf_dataset(df)
    ds = ds.map(tokenize_batch, batched=True, remove_columns=["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

if SMOKE_TEST:
    # Balanced subset so the smoke run covers both classes.
    smoke_train = (train_df.groupby("label", group_keys=False)
                   .apply(lambda g: g.sample(n=100, random_state=RANDOM_SEED)))
    smoke_val = (val_df.groupby("label", group_keys=False)
                 .apply(lambda g: g.sample(n=40, random_state=RANDOM_SEED)))
    train_ds = prep(smoke_train)
    val_ds = prep(smoke_val)
    print(f"SMOKE TEST — train={len(smoke_train)}  val={len(smoke_val)}")
else:
    train_ds = prep(train_df)
    val_ds = prep(val_df)
    print(f"FULL RUN — train={len(train_df):,}  val={len(val_df):,}")


## 5. Model, metrics, Trainer

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import roc_auc_score

def phishing_metrics(y_true, y_pred) -> dict:
    """Compute the metrics with the same definitions as src/evaluation.py."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy,
            "fpr": fpr, "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def compute_metrics(eval_pred):
    """Return phishing_metrics plus ROC-AUC for the Trainer's evaluation step."""
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=-1)
    m = phishing_metrics(labels, preds)
    try:
        m["roc_auc"] = roc_auc_score(labels, probs[:, 1])
    except Exception:
        m["roc_auc"] = float("nan")
    return m

model = AutoModelForSequenceClassification.from_pretrained(CONFIG["model_name"], num_labels=2)

# Some transformers versions reject warmup_ratio or eval_strategy, so warmup_steps is computed
# by hand and the evaluation argument name is detected at run time.
import inspect

total_steps = (len(train_ds) // CONFIG["batch_size"]) * (1 if SMOKE_TEST else CONFIG["epochs"])
warmup_steps = max(1, int(CONFIG["warmup_ratio"] * total_steps))

ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"

args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["eval_batch_size"],
    num_train_epochs=1 if SMOKE_TEST else CONFIG["epochs"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=warmup_steps,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=CONFIG["fp16"],
    seed=CONFIG["seed"],
    report_to=[],
    logging_steps=50,
    **{eval_key: "epoch"},
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


## 6. Train

With `SMOKE_TEST = True` this takes a few minutes. For the real run, set `SMOKE_TEST = False`
and re-run sections 4-5 first (about 30-45 minutes on a T4).


In [ ]:
t0 = time.time()
train_result = trainer.train()
fit_seconds = time.time() - t0
print(f"\nTraining finished in {fit_seconds/60:.1f} min ({'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'})")
print(train_result.metrics)


## 7. Evaluate on the full test set

Scores all 13,030 test emails (Table 2) and writes `SAVE_DIR/bert_uc1_full.csv`.


In [ ]:
def predict_df(df: pd.DataFrame, batch_size: int = 64):
    """Return (predicted labels, P(phishing)) for the emails in df."""
    model.eval()
    device = next(model.parameters()).device
    all_probs = []
    texts = df["text"].tolist()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, truncation=True, max_length=CONFIG["max_len"],
                             padding=True, return_tensors="pt").to(device)
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    probs = np.concatenate(all_probs, axis=0)
    preds = probs.argmax(axis=1)
    return preds, probs[:, 1]

test_eval_df = test_df if not SMOKE_TEST else test_df.sample(n=500, random_state=RANDOM_SEED)
if SMOKE_TEST:
    print("SMOKE TEST: scoring on a 500-row sample of test.csv, not the full 13,030 — "
          "numbers below are NOT the thesis results, just a pipeline check.")

t0 = time.time()
bert_preds, bert_scores = predict_df(test_eval_df)
predict_seconds = time.time() - t0

y_test = test_eval_df["label"].to_numpy(int)
full_metrics = phishing_metrics(y_test, bert_preds)
try:
    full_metrics["roc_auc"] = roc_auc_score(y_test, bert_scores)
except Exception:
    full_metrics["roc_auc"] = float("nan")

print(f"\n[bert_base] on {'500-row smoke sample' if SMOKE_TEST else 'full 13,030-row test set'} "
      f"(inference {predict_seconds:.1f}s)")
print(f"  P={full_metrics['precision']:.3f}  R={full_metrics['recall']:.3f}  "
      f"F1={full_metrics['f1']:.3f}  Acc={full_metrics['accuracy']:.3f}  "
      f"AUC={full_metrics['roc_auc']:.3f}  FPR={full_metrics['fpr']:.3f}")

bert_full_row = {
    "model": "bert_base", "precision": full_metrics["precision"], "recall": full_metrics["recall"],
    "f1": full_metrics["f1"], "accuracy": full_metrics["accuracy"], "roc_auc": full_metrics["roc_auc"],
    "fpr": full_metrics["fpr"], "tp": full_metrics["tp"], "fp": full_metrics["fp"],
    "fn": full_metrics["fn"], "tn": full_metrics["tn"], "fit_seconds": round(fit_seconds, 1),
}
pd.DataFrame([bert_full_row]).to_csv(SAVE_DIR / "bert_uc1_full.csv", index=False)
print(f"\nSaved -> {SAVE_DIR / 'bert_uc1_full.csv'}  (same schema as Thesis Codes/results/uc1_baselines.csv — "
      "append this row to that file to add the BERT row of thesis Table 2)")


## 8. Evaluate on the same 400-email sample used for GPT-4o

Reproduces `sample_test()` from `04_run_uc1_llm.py` exactly (seed 42, 200 per class), so BERT is
scored on the same emails as GPT-4o (Table 3). Writes `SAVE_DIR/bert_uc1_400.csv`.


In [ ]:
def sample_test_like_llm_runner(df: pd.DataFrame, n: int = 400, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Draw the same sample as sample_test() in 04_run_uc1_llm.py (copied line for line)."""
    te = df.copy()
    te["_row"] = te.index
    per = n // 2
    parts = [te[te.label == lab].sample(n=min(per, (te.label == lab).sum()), random_state=seed)
             for lab in (0, 1)]  # config.LEGIT, config.PHISH
    s = pd.concat(parts).sort_values("_row")
    return s

sample_400 = sample_test_like_llm_runner(test_df, n=400, seed=RANDOM_SEED)
print(f"400-sample: {len(sample_400)} emails "
      f"({(sample_400.label==1).sum()} phishing / {(sample_400.label==0).sum()} legit)")

if SMOKE_TEST:
    print("SMOKE TEST is on — this still runs against the REAL 400-sample (cheap either way), "
          "but the model itself was only trained on the tiny smoke subset, so ignore these numbers too.")

preds_400, scores_400 = predict_df(sample_400)
y_400 = sample_400["label"].to_numpy(int)
m_400 = phishing_metrics(y_400, preds_400)
try:
    m_400["roc_auc"] = roc_auc_score(y_400, scores_400)
except Exception:
    m_400["roc_auc"] = float("nan")

print(f"\n[bert_base] on the shared 400-email sample")
print(f"  P={m_400['precision']:.3f}  R={m_400['recall']:.3f}  F1={m_400['f1']:.3f}  "
      f"Acc={m_400['accuracy']:.3f}  AUC={m_400['roc_auc']:.3f}  FPR={m_400['fpr']:.3f}")

bert_400_row = {
    "model": "bert_base", "precision": m_400["precision"], "recall": m_400["recall"],
    "f1": m_400["f1"], "accuracy": m_400["accuracy"], "fpr": m_400["fpr"],
    "roc_auc": m_400["roc_auc"],
}
pd.DataFrame([bert_400_row]).to_csv(SAVE_DIR / "bert_uc1_400.csv", index=False)
print(f"\nSaved -> {SAVE_DIR / 'bert_uc1_400.csv'}  (same schema as Thesis Codes/results/uc1_llm_400.csv — "
      "append this row to that file to add the BERT row of thesis Table 3)")


## 9. Save per-email predictions

Writes `SAVE_DIR/bert_predictions.csv` in `test.csv` row order, so it lines up with
`uc1_predictions.csv`. Step 11 uses it for the BERT confusion matrix. Full run only.


In [ ]:
if not SMOKE_TEST:
    full_preds_df = pd.DataFrame({
        "label": test_df["label"].to_numpy(int),
        "bert_pred": bert_preds,
        "bert_score": np.round(bert_scores, 4),
    })
    full_preds_df.to_csv(SAVE_DIR / "bert_predictions.csv", index=False)
    print(f"Saved -> {SAVE_DIR / 'bert_predictions.csv'} (row order matches test.csv / uc1_predictions.csv)")
else:
    print("Skipped (SMOKE_TEST is on — predictions cover the 500-row sample, not the full test set).")


## 10. Save the fine-tuned model to Drive (optional)

With `SAVE_MODEL = True` the model (about 420 MB) is saved to `MODEL_DIR`. The thesis tables do not need it.


In [ ]:
SAVE_MODEL = False  # True = save the model to MODEL_DIR

if SAVE_MODEL and not SMOKE_TEST:
    save_dir = MODEL_DIR
    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(save_dir))
    tokenizer.save_pretrained(str(save_dir))
    print(f"Saved fine-tuned model -> {save_dir}")
else:
    print("Skipped model save (SAVE_MODEL is False, or this was a smoke-test model).")


## 11. Download the results

Zips the CSV files in `SAVE_DIR` and downloads them. Put them in `RESULTS_DIR`
(by default `Thesis Codes/results/`), where step 11 expects `bert_predictions.csv`.


In [ ]:
from google.colab import files

out_files = [p for p in
             [str(SAVE_DIR / "bert_uc1_full.csv"), str(SAVE_DIR / "bert_uc1_400.csv"),
              str(SAVE_DIR / "bert_predictions.csv")]
             if Path(p).exists()]

zip_path = str(SAVE_DIR / "bert_uc1_results.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    for p in out_files:
        zf.write(p, arcname=Path(p).name)

print("Zipped:", out_files)
files.download(zip_path)


## 12. Attention heatmaps (Section 4.3, Figures 5-6)

**Why attention.** The OpenAI API does not expose attention weights, so GPT-4o is explained with
SHAP (`09_explain_uc2_shap.py`). BERT runs locally, so its attention can be read directly.

**Method.** The figures show how much the `[CLS]` token attends to each word in the last layer,
averaged over the 12 heads.

**Caveat.** Attention does not always match a word's real effect (Jain & Wallace, NAACL 2019),
so the thesis pairs it with SHAP.

**Needs a full run** (`SMOKE_TEST = False`) in the same session. Figures go to `HEATMAP_DIR`.

In [ ]:
def _merge_wordpieces(tokens, weights):
    """Merge word pieces into whole words and sum their attention, to match the SHAP figures."""
    merged_tokens, merged_weights = [], []
    for tok, w in zip(tokens, weights):
        if tok.startswith("##") and merged_tokens:
            merged_tokens[-1] = merged_tokens[-1] + tok[2:]
            merged_weights[-1] = merged_weights[-1] + w
        else:
            merged_tokens.append(tok)
            merged_weights.append(w)
    return merged_tokens, merged_weights


def get_cls_attention(text: str, layer: int = -1):
    """Return (words, [CLS] attention per word, [P(legit), P(phish)]) for one email."""
    model.eval()
    device = next(model.parameters()).device
    enc = tokenizer(text, truncation=True, max_length=CONFIG["max_len"], return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**enc, output_attentions=True)
    attn_layer = out.attentions[layer]            # (1, num_heads, seq_len, seq_len)
    cls_to_all = attn_layer[0, :, 0, :].mean(dim=0)  # avg over heads -> (seq_len,)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    weights = cls_to_all.cpu().numpy().tolist()
    tokens, weights = _merge_wordpieces(tokens, weights)
    probs = torch.softmax(out.logits, dim=-1)[0].cpu().numpy()
    return tokens, weights, probs


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display


def plot_attention_bar(tokens, weights, title: str, top_n: int = 15):
    """Draw a bar chart of the top-N words by attention, styled like the SHAP bar chart."""
    weights = np.asarray(weights)
    specials = {"[CLS]", "[SEP]", "[PAD]"}
    content_idx = [i for i, t in enumerate(tokens) if t not in specials]
    top_idx = sorted(content_idx, key=lambda i: -weights[i])[:top_n]
    top_idx = sorted(top_idx, key=lambda i: weights[i])  # ascending, for a horizontal bar

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh([tokens[i] for i in top_idx], [weights[i] for i in top_idx], color="#c0392b")
    ax.set_xlabel("CLS attention weight (last layer, averaged over heads)")
    ax.set_title(title, fontsize=9, wrap=True)
    fig.tight_layout()
    return fig


def tokens_to_html(tokens, weights, title: str) -> str:
    """Return HTML with each word shaded red by its attention, like shap.plots.text()."""
    weights = np.asarray(weights)
    specials = {"[CLS]", "[SEP]", "[PAD]"}
    content_idx = [i for i, t in enumerate(tokens) if t not in specials]
    vmax = float(weights[content_idx].max()) if content_idx else float(weights.max())
    vmax = max(vmax, 1e-6)

    spans = []
    for tok, w in zip(tokens, weights):
        if tok in specials:
            continue
        alpha = 0.12 + 0.78 * min(w / vmax, 1.0)
        spans.append(
            f'<span style="background-color: rgba(192,57,43,{alpha:.2f}); '
            f'padding:1px 3px; border-radius:3px; margin:0 1px;">{tok}</span>'
        )
    body = " ".join(spans)
    return (f'<h4 style="font-family:sans-serif;">{title}</h4>'
            f'<div style="font-family:sans-serif; line-height:2.1; font-size:14px;">{body}</div>')


In [ ]:
heatmap_dir = HEATMAP_DIR
heatmap_dir.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    print("SMOKE_TEST is on -- skipping. Set SMOKE_TEST = False in the config cell, re-run cells "
          "4-6 (rebuild full datasets + fresh model) and the training cell, then come back here.")
else:
    # Use BERT's most confident phishing and legitimate emails from the 400-email sample.
    _preds_400, _scores_400 = predict_df(sample_400)
    sample_scored = sample_400.copy()
    sample_scored["bert_score"] = _scores_400  # P(phishing)

    phishing_pick = sample_scored[sample_scored.label == 1].sort_values("bert_score", ascending=False).iloc[0]
    legit_pick = sample_scored[sample_scored.label == 0].sort_values("bert_score", ascending=True).iloc[0]

    examples = [
        ("phishing_most_confident", phishing_pick),
        ("legitimate_most_confident", legit_pick),
    ]

    saved = []
    for tag, row in examples:
        tokens, weights, probs = get_cls_attention(row["text"], layer=-1)
        title = (f"{tag}  |  true label={'phishing' if row['label'] == 1 else 'legitimate'}  |  "
                 f"P(phishing)={probs[1]:.3f}  |  CLS attention, last layer, avg over heads")

        fig = plot_attention_bar(tokens, weights, title)
        png_path = heatmap_dir / f"attention_{tag}.png"
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        html = tokens_to_html(tokens, weights, title)
        html_path = heatmap_dir / f"attention_{tag}.html"
        html_path.write_text(html)
        display(HTML(html))

        print(f"Saved {png_path.name} and {html_path.name} -> {heatmap_dir}")
        saved.append((tag, str(row.get("text", ""))[:200]))

    print("\nDone -- both figures are on Drive at", heatmap_dir)
    for tag, preview in saved:
        print(f"  [{tag}] {preview}...")


In [ ]:
# Optional: also download the heatmap files (they are already in HEATMAP_DIR on Drive).
from google.colab import files as _colab_files

heatmap_files = sorted(heatmap_dir.glob("attention_*")) if heatmap_dir.exists() else []
if heatmap_files:
    zip_path = str(SAVE_DIR / "bert_attention_heatmaps.zip")
    with zipfile.ZipFile(zip_path, "w") as zf:
        for p in heatmap_files:
            zf.write(p, arcname=p.name)
    print("Zipped:", [p.name for p in heatmap_files])
    _colab_files.download(zip_path)
else:
    print("No heatmap files found yet -- run the cell above first (with SMOKE_TEST = False).")
